# 10. The benefit table: which lever wins which problem

Session 43's reframe still stands: stop asking "which lever should I
recommend" and instead build a table — rows are levers, columns are
problems — and let the columns disagree if they disagree. This notebook is
a full rewrite of the first version, not a patch.

**Why a rewrite, not a patch.** The first version of this table (this
session, before notebook 11) assembled its numbers from notebooks 06/07/08/09
— every one of which tested only `nudge_shape="uniform"` continuous ranges
against Blackout. Its headline was "Blackout wins bias, variance, saturation
AND adstock; the only real choice is whether that rigor is worth 9.9% of
revenue." Ryan asked whether notebook 06 was still pulling its weight given
it already knew something the others didn't use: `_generate_phased_schedule`
has real `nudge_shape`/`balance_signs` options, and `"uniform"` wastes about
half the negotiated band. Notebook 11 re-ran all four rigor axes plus cost
with `edge`+`balance_signs=True` added to the comparison, and it changes the
answer completely: `edge+balanced` beats Blackout on every axis notebook 11
could measure, and costs less. A follow-up in the same notebook also ruled
out `annulus+balanced` as a cheaper way to get there — it isn't.

**This notebook runs no new simulation.** Every number below is notebook
11's own real, committed, `nbformat`-validated output (its "Bringing it
together" table), copied verbatim. The old version's caveat about notebook
07's window-standardisation gap no longer applies — notebook 11 uses the
fix throughout. The old version's `TV alone +/-80%` diagnostic row (isolating
one channel, from notebook 07) is dropped: notebook 11 never tested it, and
the whole-plan, all-channels-phased-together comparison is now the standard
this project measures against.

## The table — from notebook 11

Five levers, the same set notebook 11 settled on: the historical default
(`+/-80%` continuous, `uniform` shape), the two `edge`+`balance_signs=True`
points notebook 06 flagged as efficient (`+/-40%` and `+/-80%`), the
`annulus`+`balance_signs=True` follow-up that checked whether the *middle*
shape was a better trade, and Blackout. All five numbers per lever —
bias-removed %, CV-narrowed %, saturation `b_sd`-narrowed % (averaged across
both truths notebook 09 tests), adstock gain-surviving % at decay=0.7, and
whole-plan revenue cost at `b=0.6` — are notebook 11's own committed cells,
not recomputed here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# notebook 11, cell 18's real committed output ("Bringing it together")
LEVER_ORDER = [
    "+/-80% (uniform)",
    "+/-40% (edge, balanced)",
    "+/-80% (edge, balanced)",
    "+/-80% (annulus, balanced)",
    "Blackout",
]

benefit_table = pd.DataFrame(
    {
        "bias: removed %": {
            "+/-80% (uniform)": 58.21,
            "+/-40% (edge, balanced)": 48.65,
            "+/-80% (edge, balanced)": 81.18,
            "+/-80% (annulus, balanced)": 75.55,
            "Blackout": 74.11,
        },
        "variance: CV narrowed %": {
            "+/-80% (uniform)": 57.18,
            "+/-40% (edge, balanced)": 56.85,
            "+/-80% (edge, balanced)": 72.25,
            "+/-80% (annulus, balanced)": 66.44,
            "Blackout": 62.86,
        },
        "saturation: b_sd narrowed %, avg both truths": {
            "+/-80% (uniform)": 14.6,
            "+/-40% (edge, balanced)": 10.6,
            "+/-80% (edge, balanced)": 45.7,
            "+/-80% (annulus, balanced)": 26.5,
            "Blackout": 45.4,
        },
        "adstock: % of gain surviving at decay=0.7": {
            "+/-80% (uniform)": 74.34,
            "+/-40% (edge, balanced)": 79.04,
            "+/-80% (edge, balanced)": 90.43,
            "+/-80% (annulus, balanced)": 70.89,
            "Blackout": 80.74,
        },
        "cost: revenue given up %": {
            "+/-80% (uniform)": 2.285,
            "+/-40% (edge, balanced)": 1.808,
            "+/-80% (edge, balanced)": 8.885,
            "+/-80% (annulus, balanced)": 4.675,
            "Blackout": 9.915,
        },
    }
).loc[LEVER_ORDER]

pd.set_option("display.width", 220)
print(benefit_table.round(1).to_string())

print("\nbest lever per column:")
for col in [
    "bias: removed %",
    "variance: CV narrowed %",
    "saturation: b_sd narrowed %, avg both truths",
    "adstock: % of gain surviving at decay=0.7",
]:
    print(f"  {col:<48} -> {benefit_table[col].idxmax()}")
print(
    f"  {'cost: revenue given up %':<48} -> "
    f"{benefit_table['cost: revenue given up %'].idxmin()} (lowest cost)"
)

**`+/-80% (edge, balanced)` wins every rigor column, and it isn't the
expensive option either.** This is not "one lever wins rigor, another wins
cost" — the old version's story about Blackout — and it is not a four-way
split across trade-offs as session 43's original design anticipated. One
lever dominates outright: best bias removal, best variance narrowing, best
saturation identification, best adstock robustness, *and* cheaper than
Blackout (8.89% vs 9.92% of revenue). The only lever cheaper than `edge+80%`
is `edge+40%`, which is itself part of the same family (same shape, smaller
cap) — buying most of `uniform+80%`'s rigor at 80% of its cost.

**`annulus+balanced` is not the frontier point it looked like.** Notebook 06
found `annulus` more cost-efficient than `edge` on variance alone (TV-only).
That held up here for cost — `annulus+80%` costs about half of `edge+80%`
(4.68% vs 8.89%) — but it does not hold up for rigor. `annulus+80%` is a
distant second to `edge`/Blackout on saturation (26.5% vs 45.7%/45.4%) and
is the *worst* lever of the five on adstock survival at decay=0.7 (70.9%) —
worse than plain `uniform+80%` (74.3%). Buying cheap variance/bias
improvement with `annulus` quietly gives up the saturation and adstock
rigor that `edge` gets for free.

**The old "Blackout wins everything" headline was real numbers, wrong
comparison.** Every rigor column really did point to Blackout — as long as
the only continuous option on the table was the wasteful `uniform` draw.
Once `edge+balanced` is on the table, Blackout is not just expensive, it is
dominated: there is no axis on which Blackout still wins.

## A picture

Four rigor columns, one bar chart each, plus cost on its own log-scale
panel — same lever order, same colour per lever throughout, so "the same
bar wins every rigor panel, and it isn't the tallest cost bar" is a shape,
not a sentence.

In [ ]:
colors = {
    "+/-80% (uniform)": "#9ca3af",
    "+/-40% (edge, balanced)": "#60a5fa",
    "+/-80% (edge, balanced)": "#059669",
    "+/-80% (annulus, balanced)": "#f59e0b",
    "Blackout": "#111827",
}

panels = [
    "bias: removed %",
    "variance: CV narrowed %",
    "saturation: b_sd narrowed %, avg both truths",
    "adstock: % of gain surviving at decay=0.7",
]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=False)
for ax, col in zip(axes, panels):
    vals = benefit_table.loc[LEVER_ORDER, col]
    ax.bar(LEVER_ORDER, vals, color=[colors[lv] for lv in LEVER_ORDER])
    ax.set_title(col, fontsize=8)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.set_ylim(0, 100)
fig.suptitle("Every rigor column, same winner: +/-80% (edge, balanced)")
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(6, 3.6))
vals = benefit_table.loc[LEVER_ORDER, "cost: revenue given up %"]
ax2.bar(LEVER_ORDER, vals, color=[colors[lv] for lv in LEVER_ORDER])
ax2.set_title(
    "cost: revenue given up % (log scale) -- the rigor winner isn't the cost loser"
)
ax2.set_yscale("log")
ax2.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout()
plt.show()

## Bringing it together

The table is simpler than session 43's design anticipated, and simpler
still than the first version of this notebook concluded. It is not four
columns pulling in different directions (session 43), and it is not one
lever winning rigor while a cheaper one wins cost (this notebook's own first
version, about Blackout). One lever — `+/-80% (edge, balanced)` — wins every
column, cost included. Three consequences follow, none implemented here:

- **`recommend_levers()`** currently makes a per-channel Blackout-vs-
  continuous call via a CV-improvement threshold. That comparison is now
  wrong on both sides: Blackout is dominated, not a genuine rigor winner
  paid for in cost; and the "continuous" side of the comparison isn't fixed
  to a shape at all — it silently inherits whatever `nudge_shape` the
  caller happened to construct the `BudgetPhaser` with, which defaults to
  `"uniform"`, the shape this notebook just retired. Open question, not yet
  decided: should the fix be as simple as changing that default, or should
  `recommend_levers()`'s own output start naming a shape/strategy
  explicitly rather than assuming the caller already set the right one
  upstream? (Tracked as item 11b.)
- **`overview.html` section 4** currently frames phasing as a variance-only
  search that lands on Blackout. Both the lever it picks and the one-
  dimensional framing need to change — tracked for the page's broader
  rewrite (item 11c), not touched here.
- **Neither of the above should be implemented by re-deriving numbers from
  this notebook.** Notebook 11 is the source of truth for method,
  verification, and the `annulus` follow-up; this notebook is the one-page
  answer for anyone who wants the table without the method.